In [6]:
# Warping
import cv2
import numpy as np

# Setup ArUco
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250)
parameters = cv2.aruco.DetectorParameters()
parameters.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

alpha = 0.05          # smoothing factor: lower = smoother but more lag
smooth_pts = None    # will hold the smoothed corner positions

while True:
    ret, frame = cap.read()

    if not ret:             # If we can't read from the webcam, exit the loop
        break

    h, w = frame.shape[:2] 
    corners, ids, _ = cv2.aruco.detectMarkers(frame, aruco_dict, parameters=parameters)

    if ids is not None:

        cv2.aruco.drawDetectedMarkers(frame, corners, ids)
        
        # 1. Get the 4 corners of the first detected marker
        # corners[0] is shape (1, 4, 2) -> we reshape to (4, 2)
        src_pts = corners[0].reshape(4, 2).astype(np.float32)

        # Apply exponential moving average to stabilize corners
        if smooth_pts is None:
            smooth_pts = src_pts
        else:
            smooth_pts = alpha * src_pts + (1 - alpha) * smooth_pts

        # 2. Define where we want these corners to go (a 200x200 square)

        x0 = w - 650
        y0 = h - 400
        dst_pts = np.float32([
            [x0, y0],
            [x0 + 150, y0],
            [x0 + 150, y0 + 150],
            [x0, y0 + 150]
        ])


        # 3. Calculate the Transformation Matrix
        matrix = cv2.getPerspectiveTransform(smooth_pts, dst_pts)

        # 4. Apply the Warp
        warped = cv2.warpPerspective(frame, matrix, (w, h))

        cv2.imshow("Warped Marker", warped)

    cv2.imshow("Original", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()